<a href="https://colab.research.google.com/github/anshumandwivediusa/mcp/blob/main/codes/mcp_stdio_transport.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP over stdio — Minimal Worked Example

This notebook demonstrates the **stdio transport** from MCP's Core Architecture module: a client spawns a server as a local subprocess and exchanges newline-delimited JSON-RPC messages over its stdin/stdout streams — no network, no ports, no auth.

**What happens here:**
1. `server.py` is written to disk (a tiny MCP server exposing one tool: `add`)
2. `client.py`'s logic spawns that script as a **child process**
3. The client sends an `initialize` handshake, then calls the `add` tool
4. Every message exchanged is printed so you can see the raw JSON-RPC traffic

**Run all cells in order** (Runtime → Run all). Pure Python standard library only — no installs needed.

## 1. Write the Server

`%%writefile` saves this cell's contents to `server.py` on disk, since the client needs an actual file path to spawn as a subprocess — it can't spawn a function living in this notebook's memory.

The server is intentionally minimal: it reads one JSON-RPC message per line from stdin, handles two methods (`initialize` and `tools/call` for a tool named `add`), and writes one JSON-RPC response per line to stdout.

In [ ]:
%%writefile server.py
# server.py
import sys, json

def handle(msg):
    if msg["method"] == "initialize":
        return {"jsonrpc": "2.0", "id": msg["id"],
                "result": {"capabilities": {"tools": {}}}}
    if msg["method"] == "tools/call" and msg["params"]["name"] == "add":
        a, b = msg["params"]["arguments"]["a"], msg["params"]["arguments"]["b"]
        return {"jsonrpc": "2.0", "id": msg["id"],
                "result": {"content": [{"type": "text", "text": str(a + b)}]}}

for line in sys.stdin:
    request = json.loads(line)
    response = handle(request)
    if response:
        print(json.dumps(response), flush=True)   # one JSON object per line

Quick sanity check — confirm the file was written correctly before spawning it:

In [ ]:
!cat server.py

## 2. Spawn the Server and Define `send()`

This is the client side: `subprocess.Popen` launches `server.py` as a **child process** of this notebook's Python kernel, with `stdin=PIPE` and `stdout=PIPE` giving us direct read/write access to its input and output streams.

`send()` is the core mechanic of stdio transport: write one JSON-RPC message + newline to the server's stdin, flush immediately so it isn't buffered, then block on `readline()` until the server writes back its one-line JSON-RPC response.

In [ ]:
# client.py
import subprocess, json

proc = subprocess.Popen(
    ["python3", "server.py"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    text=True
)

def send(msg):
    proc.stdin.write(json.dumps(msg) + "\n")
    proc.stdin.flush()
    return json.loads(proc.stdout.readline())

print(f"Server subprocess spawned with PID: {proc.pid}")

## 3. Capability Handshake

Sends the `initialize` request first, matching the negotiation sequence from Module 2 — the server replies with its declared capabilities (here, just `{"tools": {}}`, signaling it supports the tools primitive).

In [ ]:
# 1. Handshake
response = send({"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {}})
print(response)

## 4. Call the `add` Tool

Sends a `tools/call` request for `add(a=4, b=7)`. The server computes `4 + 7` and returns the result wrapped in the standard MCP tool-result content format.

In [ ]:
# 2. Call the tool
response = send({
    "jsonrpc": "2.0", "id": 2, "method": "tools/call",
    "params": {"name": "add", "arguments": {"a": 4, "b": 7}}
})
print(response)

## 5. Clean Up the Subprocess

Real MCP clients terminate the server subprocess on disconnect — this cell does that explicitly, illustrating the "lifecycle tied to the client" property: this server has no independent existence once the client is done with it.

In [ ]:
proc.terminate()
proc.wait()
print(f"Server subprocess {proc.pid} terminated.")

## Notes

- This uses `%%writefile` because a subprocess needs a real file on disk to execute — you can't `Popen` a Python function living in the notebook's memory.
- No network calls, no ports, no auth headers anywhere in this notebook — everything happens through OS-level pipes between two local processes, exactly as described in the stdio transport section.
- A production MCP server would use the official MCP SDK rather than hand-rolled `sys.stdin` parsing (it handles schema validation, the full `initialize` capability payload, and proper JSON-RPC error responses for you) — but the underlying mechanics shown here (spawn → write a line → read a line) are exactly what's happening underneath the SDK too.